# Polymarket Buyer Risk Scoring Using Orderbook Anomaly Detection

This notebook builds a Spark-based MVP pipeline for Polymarket orderbook data.

The goal is to flag markets with **buyer-risk / manipulation-like orderbook behavior** using explainable signals such as:

- bid-ask spread spikes
- spread instability
- price jumps
- activity spikes
- large order changes
- buy/sell imbalance
- liquidity stress

The score does **not** prove fraud or manipulation. It is a screening tool for markets that may require further review.

## 1. Imports and Spark Setup

Run this section first. It creates the Spark session and imports all libraries used later in the notebook.

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from functools import reduce
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName("Polymarket Buyer Risk Scoring")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark

ConnectionRefusedError: [Errno 111] Connection refused

## 2. Path Configuration

For the MVP, we use one orderbook file first. Once the pipeline works end-to-end, the same logic can be expanded to multiple orderbook files.

In [2]:
BASE_PATH = "/mnt/data/public/polymarket"

LABELS_PATH = f"{BASE_PATH}/labels/market_targets.parquet"
ORDERBOOK_PATH = f"{BASE_PATH}/orderbook/orderbook_2026-03-06.parquet"

# Use a relative output path so Spark can create the folder in the working directory.
OUTPUT_PATH = "polymarket_outputs"

print("Labels path:", LABELS_PATH)
print("Orderbook path:", ORDERBOOK_PATH)
print("Output path:", OUTPUT_PATH)

Labels path: /mnt/data/public/polymarket/labels/market_targets.parquet
Orderbook path: /mnt/data/public/polymarket/orderbook/orderbook_2026-03-06.parquet
Output path: polymarket_outputs


## 3. Load Data and Inspect Schemas

This section loads the market metadata and one day of tick-level orderbook data.

In [3]:
df_labels = spark.read.parquet(LABELS_PATH)
df_ob = spark.read.parquet(ORDERBOOK_PATH)

print("Labels schema:")
df_labels.printSchema()

print("Orderbook schema:")
df_ob.printSchema()

df_labels.show(5, truncate=False)
df_ob.show(5, truncate=False)

Labels schema:
root
 |-- condition_id: string (nullable = true)
 |-- question: string (nullable = true)
 |-- category: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- closed: boolean (nullable = true)
 |-- uma_status: string (nullable = true)
 |-- volume: double (nullable = true)
 |-- liquidity: double (nullable = true)
 |-- clob_token_id_yes: string (nullable = true)
 |-- clob_token_id_no: string (nullable = true)
 |-- target: byte (nullable = true)

Orderbook schema:
root
 |-- timestamp_received: long (nullable = true)
 |-- timestamp_created_at: long (nullable = true)
 |-- market_id: string (nullable = true)
 |-- best_bid: float (nullable = true)
 |-- best_ask: float (nullable = true)
 |-- change_price: float (nullable = true)
 |-- change_size: float (nullable = true)
 |-- change_side: string (nullable = true)
 |-- token_id: string (nullable = true)
 |-- spread: float (nullable = true)
 |-- mid_price: float (nullable = true)

+-----------------------------------

In [4]:
# Basic counts

print("Labels rows:", df_labels.count())
print("Orderbook rows:", df_ob.count())

df_labels.select(
    F.countDistinct("condition_id").alias("distinct_condition_ids"),
    F.countDistinct("question").alias("distinct_questions")
).show()

df_ob.select(
    F.countDistinct("market_id").alias("distinct_market_ids"),
    F.countDistinct("token_id").alias("distinct_token_ids")
).show()

Labels rows: 123895
Orderbook rows: 325974151
+----------------------+------------------+
|distinct_condition_ids|distinct_questions|
+----------------------+------------------+
|                123895|            102476|
+----------------------+------------------+

+-------------------+------------------+
|distinct_market_ids|distinct_token_ids|
+-------------------+------------------+
|              24465|             24465|
+-------------------+------------------+



## 4. Data Quality Checks

Check missing values in important columns before joining and feature engineering.

In [5]:
df_labels.select(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("condition_id").isNull(), 1).otherwise(0)).alias("null_condition_id"),
    F.sum(F.when(F.col("question").isNull(), 1).otherwise(0)).alias("null_question"),
    F.sum(F.when((F.col("category").isNull()) | (F.trim(F.col("category")) == ""), 1).otherwise(0)).alias("blank_category"),
    F.sum(F.when(F.col("volume").isNull(), 1).otherwise(0)).alias("null_volume"),
    F.sum(F.when(F.col("liquidity").isNull(), 1).otherwise(0)).alias("null_liquidity")
).show()

df_ob.select(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("market_id").isNull(), 1).otherwise(0)).alias("null_market_id"),
    F.sum(F.when(F.col("token_id").isNull(), 1).otherwise(0)).alias("null_token_id"),
    F.sum(F.when(F.col("best_bid").isNull(), 1).otherwise(0)).alias("null_best_bid"),
    F.sum(F.when(F.col("best_ask").isNull(), 1).otherwise(0)).alias("null_best_ask"),
    F.sum(F.when(F.col("spread").isNull(), 1).otherwise(0)).alias("null_spread"),
    F.sum(F.when(F.col("mid_price").isNull(), 1).otherwise(0)).alias("null_mid_price")
).show()

+------+-----------------+-------------+--------------+-----------+--------------+
|  rows|null_condition_id|null_question|blank_category|null_volume|null_liquidity|
+------+-----------------+-------------+--------------+-----------+--------------+
|123895|                0|            0|        123895|          0|             0|
+------+-----------------+-------------+--------------+-----------+--------------+

+---------+--------------+-------------+-------------+-------------+-----------+--------------+
|     rows|null_market_id|null_token_id|null_best_bid|null_best_ask|null_spread|null_mid_price|
+---------+--------------+-------------+-------------+-------------+-----------+--------------+
|325974151|             0|            0|            0|            0|          0|             0|
+---------+--------------+-------------+-------------+-------------+-----------+--------------+



## 5. Validate Join Strategy

The orderbook uses `market_id`, while the labels table uses `condition_id`. This section checks whether they match.

A token-based check is also included to confirm whether `token_id` improves coverage.

In [6]:
# Direct market ID match: orderbook.market_id = labels.condition_id

df_ob_markets = df_ob.select("market_id").distinct()

df_label_ids = df_labels.select(
    F.col("condition_id").alias("market_id"),
    "question",
    "category",
    "volume",
    "liquidity",
    "clob_token_id_yes",
    "clob_token_id_no"
)

df_join_test = df_ob_markets.join(df_label_ids, on="market_id", how="left")

df_join_test.select(
    F.count("*").alias("distinct_orderbook_markets"),
    F.count("question").alias("matched_to_labels"),
    (F.count("question") / F.count("*")).alias("match_rate")
).show()

+--------------------------+-----------------+------------------+
|distinct_orderbook_markets|matched_to_labels|        match_rate|
+--------------------------+-----------------+------------------+
|                     24465|            12665|0.5176783159615778|
+--------------------------+-----------------+------------------+



In [7]:
# Token-based match check

df_label_tokens = (
    df_labels
    .select(
        "condition_id",
        "question",
        "category",
        "volume",
        "liquidity",
        F.col("clob_token_id_yes").alias("token_id"),
        F.lit("YES").alias("outcome_side")
    )
    .unionByName(
        df_labels.select(
            "condition_id",
            "question",
            "category",
            "volume",
            "liquidity",
            F.col("clob_token_id_no").alias("token_id"),
            F.lit("NO").alias("outcome_side")
        )
    )
)

df_token_join_test = (
    df_ob.select("market_id", "token_id").distinct()
    .join(df_label_tokens, on="token_id", how="left")
)

df_market_match_by_token = (
    df_token_join_test
    .groupBy("market_id")
    .agg(
        F.max(F.when(F.col("question").isNotNull(), 1).otherwise(0)).alias("has_token_match")
    )
)

df_market_match_by_token.select(
    F.count("*").alias("distinct_orderbook_markets"),
    F.sum("has_token_match").alias("markets_matched_by_token"),
    (F.sum("has_token_match") / F.count("*")).alias("market_token_match_rate")
).show()

+--------------------------+------------------------+-----------------------+
|distinct_orderbook_markets|markets_matched_by_token|market_token_match_rate|
+--------------------------+------------------------+-----------------------+
|                     24465|                   12665|     0.5176783159615778|
+--------------------------+------------------------+-----------------------+



### Join Decision

The MVP keeps only markets that successfully join to the labels table. This is necessary so the final output has interpretable market questions, volume, liquidity, and topic information.

In [13]:
# Create market metadata table

df_labels_meta = df_labels.select(
    F.col("condition_id").alias("market_id"),
    "question",
    "category",
    "end_date",
    "closed",
    "uma_status",
    F.coalesce(F.col("volume"), F.lit(0.0)).alias("volume"),
    F.coalesce(F.col("liquidity"), F.lit(0.0)).alias("liquidity"),
    "clob_token_id_yes",
    "clob_token_id_no",
    "target"
)

# Inner join retains only interpretable markets
df_ob_labeled = (
    df_ob
    .join(df_labels_meta, on="market_id", how="inner")
    .withColumn(
        "outcome_side",
        F.when(F.col("token_id") == F.col("clob_token_id_yes"), F.lit("YES"))
         .when(F.col("token_id") == F.col("clob_token_id_no"), F.lit("NO"))
         .otherwise(F.lit("UNKNOWN"))
    )
)

df_ob_labeled.select(
    "market_id",
    "question",
    "category",
    "outcome_side",
    "change_side",
    "best_bid",
    "best_ask",
    "spread",
    "mid_price",
    "change_price",
    "change_size",
    "volume",
    "liquidity"
).show(10, truncate=False)

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
# Check how much orderbook data remains after joining with labels

total_ob_rows = df_ob.count()
labeled_ob_rows = df_ob_labeled.count()

print("Total orderbook rows:", total_ob_rows)
print("Labeled orderbook rows:", labeled_ob_rows)
print("Row retention rate:", labeled_ob_rows / total_ob_rows if total_ob_rows > 0 else None)

## 6. Clean Category and Convert Timestamp

The original category field is often blank, so we create a cleaned category column. We also convert the millisecond timestamp to a Spark timestamp.

In [ ]:
df_ob_labeled = (
    df_ob_labeled
    .withColumn(
        "category_clean",
        F.when(
            (F.col("category").isNull()) | (F.trim(F.col("category")) == ""),
            F.lit("uncategorized")
        ).otherwise(F.col("category"))
    )
    .withColumn(
        "is_unresolved",
        F.when(F.col("closed") == False, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "event_time",
        F.to_timestamp(
            F.from_unixtime((F.col("timestamp_received") / F.lit(1000)).cast("long"))
        )
    )
)

df_ob_labeled.select(
    "timestamp_received",
    "event_time",
    "market_id",
    "question",
    "best_bid",
    "best_ask",
    "spread",
    "mid_price"
).show(10, truncate=False)

df_ob_labeled.groupBy("category_clean").count().orderBy(F.desc("count")).show(20, truncate=False)

## 7. Active Market Filtering

Inactive or extremely thin markets can create noisy signals. This section keeps markets with sufficient activity, volume, and liquidity.

In [ ]:
# Labeled data size and time range

df_ob_labeled.select(
    F.count("*").alias("rows"),
    F.countDistinct("market_id").alias("distinct_markets"),
    F.min("event_time").alias("min_event_time"),
    F.max("event_time").alias("max_event_time")
).show(truncate=False)

In [ ]:
# Market-level activity summary

df_market_activity = (
    df_ob_labeled
    .filter(F.col("event_time").isNotNull())
    .groupBy(
        "market_id",
        "question",
        "category_clean",
        "closed",
        "volume",
        "liquidity"
    )
    .agg(
        F.count("*").alias("tick_count"),
        F.min("event_time").alias("first_seen"),
        F.max("event_time").alias("last_seen"),
        F.avg("spread").alias("avg_spread"),
        F.expr("percentile_approx(spread, 0.5)").alias("median_spread"),
        F.avg("mid_price").alias("avg_mid_price")
    )
)

df_market_activity.orderBy(F.desc("tick_count")).show(10, truncate=False)

In [ ]:
# Compute thresholds for filtering active markets

volume_bounds = df_market_activity.approxQuantile("volume", [0.50, 0.95, 0.9975], 0.01)
liquidity_bounds = df_market_activity.approxQuantile("liquidity", [0.50, 0.95, 0.9975], 0.01)
tick_bounds = df_market_activity.approxQuantile("tick_count", [0.50, 0.75, 0.90], 0.01)

print("Volume percentiles [50%, 95%, 99.75%]:", volume_bounds)
print("Liquidity percentiles [50%, 95%, 99.75%]:", liquidity_bounds)
print("Tick count percentiles [50%, 75%, 90%]:", tick_bounds)

In [ ]:
# Filter active and unresolved markets

min_volume = volume_bounds[0]
min_liquidity = liquidity_bounds[0]
min_tick_count = 100

df_active_markets = (
    df_market_activity
    .filter(F.col("closed") == False)
    .filter(F.col("volume") >= min_volume)
    .filter(F.col("liquidity") >= min_liquidity)
    .filter(F.col("tick_count") >= min_tick_count)
)

df_active_markets.select(
    F.count("*").alias("active_markets")
).show()

df_active_markets.orderBy(F.desc("tick_count")).show(10, truncate=False)

In [ ]:
# Keep only active orderbook rows

df_ob_active = (
    df_ob_labeled
    .join(
        df_active_markets.select("market_id"),
        on="market_id",
        how="inner"
    )
)

df_ob_active.select(
    F.count("*").alias("active_orderbook_rows"),
    F.countDistinct("market_id").alias("active_orderbook_markets")
).show()

## 8. Market-Time Window Feature Engineering

This section aggregates tick-level orderbook data into one row per market per 1-hour window.

In [ ]:
# Persist active data because it is reused in multiple steps

df_ob_active = df_ob_active.repartition("market_id").persist()

# Materialize cache
df_ob_active.count()

In [ ]:
# Create 1-hour time windows

df_window_base = (
    df_ob_active
    .filter(F.col("event_time").isNotNull())
    .filter(F.col("spread").isNotNull())
    .filter(F.col("mid_price").isNotNull())
    .withColumn("time_window", F.window("event_time", "1 hour"))
    .withColumn("window_start", F.col("time_window.start"))
    .withColumn("window_end", F.col("time_window.end"))
)

In [ ]:
# Aggregate tick-level rows into market-window features

df_features = (
    df_window_base
    .groupBy(
        "market_id",
        "question",
        "category_clean",
        "window_start",
        "window_end"
    )
    .agg(
        # Activity
        F.count("*").alias("tick_count"),
        F.countDistinct("token_id").alias("distinct_tokens"),

        # Spread behavior
        F.avg("spread").alias("avg_spread"),
        F.expr("percentile_approx(spread, 0.5)").alias("median_spread"),
        F.max("spread").alias("max_spread"),
        F.stddev("spread").alias("spread_volatility"),

        # Price behavior
        F.avg("mid_price").alias("avg_mid_price"),
        F.min("mid_price").alias("min_mid_price"),
        F.max("mid_price").alias("max_mid_price"),
        F.stddev("mid_price").alias("mid_price_volatility"),

        # Orderbook change behavior
        F.sum("change_size").alias("total_change_size"),
        F.avg("change_size").alias("avg_change_size"),
        F.max("change_size").alias("max_change_size"),

        # Buy / sell pressure
        F.sum(
            F.when(F.col("change_side") == "BUY", F.col("change_size")).otherwise(0)
        ).alias("buy_pressure"),

        F.sum(
            F.when(F.col("change_side") == "SELL", F.col("change_size")).otherwise(0)
        ).alias("sell_pressure")
    )
    .withColumn(
        "price_range",
        F.col("max_mid_price") - F.col("min_mid_price")
    )
    .withColumn(
        "total_pressure",
        F.col("buy_pressure") + F.col("sell_pressure")
    )
    .withColumn(
        "imbalance_score",
        F.when(
            F.col("total_pressure") > 0,
            F.abs(F.col("buy_pressure") - F.col("sell_pressure")) / F.col("total_pressure")
        ).otherwise(0)
    )
)

df_features = (
    df_features
    .fillna({
        "spread_volatility": 0.0,
        "mid_price_volatility": 0.0,
        "total_change_size": 0.0,
        "avg_change_size": 0.0,
        "max_change_size": 0.0,
        "buy_pressure": 0.0,
        "sell_pressure": 0.0,
        "total_pressure": 0.0,
        "imbalance_score": 0.0
    })
)

df_features.select(
    "market_id",
    "question",
    "category_clean",
    "window_start",
    "window_end",
    "tick_count",
    "avg_spread",
    "median_spread",
    "spread_volatility",
    "price_range",
    "total_change_size",
    "buy_pressure",
    "sell_pressure",
    "imbalance_score"
).orderBy(F.desc("tick_count")).show(10, truncate=False)

df_features.select(
    F.count("*").alias("market_window_rows"),
    F.countDistinct("market_id").alias("markets")
).show()

## 9. Anomaly Flag Creation

This section converts market-window features into explainable anomaly flags using percentile thresholds and simple rules.

In [ ]:
df_features = df_features.persist()
df_features.count()

In [ ]:
# Compute percentile thresholds for anomaly flags

feature_cols = [
    "avg_spread",
    "spread_volatility",
    "price_range",
    "tick_count",
    "max_change_size",
    "total_change_size",
    "imbalance_score"
]

quantile_probs = [0.25, 0.50, 0.75, 0.95]

quantile_values = df_features.approxQuantile(feature_cols, quantile_probs, 0.01)

thresholds = {
    col: {
        "q25": vals[0],
        "q50": vals[1],
        "q75": vals[2],
        "q95": vals[3]
    }
    for col, vals in zip(feature_cols, quantile_values)
}

thresholds

In [ ]:
# Create explainable anomaly flags

df_flags = (
    df_features
    .withColumn(
        "spread_spike_flag",
        F.when(F.col("avg_spread") >= F.lit(thresholds["avg_spread"]["q95"]), 1).otherwise(0)
    )
    .withColumn(
        "spread_instability_flag",
        F.when(F.col("spread_volatility") >= F.lit(thresholds["spread_volatility"]["q95"]), 1).otherwise(0)
    )
    .withColumn(
        "price_jump_flag",
        F.when(F.col("price_range") >= F.lit(thresholds["price_range"]["q95"]), 1).otherwise(0)
    )
    .withColumn(
        "activity_spike_flag",
        F.when(F.col("tick_count") >= F.lit(thresholds["tick_count"]["q95"]), 1).otherwise(0)
    )
    .withColumn(
        "large_order_flag",
        F.when(F.col("max_change_size") >= F.lit(thresholds["max_change_size"]["q95"]), 1).otherwise(0)
    )
    .withColumn(
        "high_volume_change_flag",
        F.when(F.col("total_change_size") >= F.lit(thresholds["total_change_size"]["q95"]), 1).otherwise(0)
    )
    .withColumn(
        "imbalance_flag",
        F.when(
            (F.col("imbalance_score") >= F.lit(0.80)) &
            (F.col("total_pressure") > 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "liquidity_stress_flag",
        F.when(
            (F.col("avg_spread") >= F.lit(thresholds["avg_spread"]["q75"])) &
            (F.col("tick_count") <= F.lit(thresholds["tick_count"]["q25"])),
            1
        ).otherwise(0)
    )
)

df_flags.select(
    F.sum("spread_spike_flag").alias("spread_spike_windows"),
    F.sum("spread_instability_flag").alias("spread_instability_windows"),
    F.sum("price_jump_flag").alias("price_jump_windows"),
    F.sum("activity_spike_flag").alias("activity_spike_windows"),
    F.sum("large_order_flag").alias("large_order_windows"),
    F.sum("high_volume_change_flag").alias("high_volume_change_windows"),
    F.sum("imbalance_flag").alias("imbalance_windows"),
    F.sum("liquidity_stress_flag").alias("liquidity_stress_windows")
).show()

In [ ]:
# Count total anomaly flags per market-window

flag_cols = [
    "spread_spike_flag",
    "spread_instability_flag",
    "price_jump_flag",
    "activity_spike_flag",
    "large_order_flag",
    "high_volume_change_flag",
    "imbalance_flag",
    "liquidity_stress_flag"
]

df_flags = df_flags.withColumn(
    "anomaly_count",
    reduce(lambda a, b: a + b, [F.col(c) for c in flag_cols])
)

df_flags.select(
    "market_id",
    "question",
    "category_clean",
    "window_start",
    "window_end",
    "tick_count",
    "avg_spread",
    "spread_volatility",
    "price_range",
    "imbalance_score",
    "anomaly_count"
).orderBy(F.desc("anomaly_count"), F.desc("tick_count")).show(20, truncate=False)

## 10. Buyer Risk / Manipulation-Likelihood Score

This section combines anomaly flags into a 0–100 score.

The score does not prove manipulation. It ranks market-windows based on observed abnormal orderbook behavior.

In [ ]:
df_scored = (
    df_flags
    .withColumn(
        "risk_score",
        (
            F.col("spread_spike_flag") * 20 +
            F.col("spread_instability_flag") * 15 +
            F.col("price_jump_flag") * 20 +
            F.col("activity_spike_flag") * 15 +
            F.col("large_order_flag") * 10 +
            F.col("high_volume_change_flag") * 10 +
            F.col("imbalance_flag") * 5 +
            F.col("liquidity_stress_flag") * 5
        )
    )
    .withColumn(
        "risk_band",
        F.when(F.col("risk_score") >= 75, "Very High")
         .when(F.col("risk_score") >= 50, "High")
         .when(F.col("risk_score") >= 25, "Moderate")
         .otherwise("Low")
    )
    .withColumn(
        "risk_drivers",
        F.concat_ws(
            ", ",
            F.when(F.col("spread_spike_flag") == 1, "spread spike"),
            F.when(F.col("spread_instability_flag") == 1, "spread instability"),
            F.when(F.col("price_jump_flag") == 1, "price jump"),
            F.when(F.col("activity_spike_flag") == 1, "activity spike"),
            F.when(F.col("large_order_flag") == 1, "large order"),
            F.when(F.col("high_volume_change_flag") == 1, "high volume change"),
            F.when(F.col("imbalance_flag") == 1, "buy/sell imbalance"),
            F.when(F.col("liquidity_stress_flag") == 1, "liquidity stress")
        )
    )
)

df_scored.select(
    "market_id",
    "question",
    "category_clean",
    "window_start",
    "window_end",
    "tick_count",
    "avg_spread",
    "spread_volatility",
    "price_range",
    "imbalance_score",
    "anomaly_count",
    "risk_score",
    "risk_band",
    "risk_drivers"
).orderBy(F.desc("risk_score"), F.desc("anomaly_count")).show(20, truncate=False)

## 11. Market-Level Ranking

This section aggregates market-window risk scores into one row per market.

In [ ]:
df_market_scores = (
    df_scored
    .groupBy(
        "market_id",
        "question",
        "category_clean"
    )
    .agg(
        F.count("*").alias("window_count"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.max("risk_score").alias("max_risk_score"),
        F.sum(F.when(F.col("risk_score") >= 75, 1).otherwise(0)).alias("very_high_risk_windows"),
        F.sum(F.when(F.col("risk_score") >= 50, 1).otherwise(0)).alias("high_risk_windows"),
        F.avg("avg_spread").alias("overall_avg_spread"),
        F.max("price_range").alias("max_price_range"),
        F.avg("imbalance_score").alias("avg_imbalance_score"),
        F.sum("anomaly_count").alias("total_anomaly_count")
    )
    .withColumn(
        "high_risk_window_rate",
        F.col("high_risk_windows") / F.col("window_count")
    )
)

df_market_scores.orderBy(
    F.desc("max_risk_score"),
    F.desc("high_risk_windows"),
    F.desc("avg_risk_score")
).show(20, truncate=False)

## 12. Keyword-Based Topic Classification

The original category field is mostly blank, so this section classifies each market using keyword rules from the market question.

In [ ]:
df_market_scores_categorized = (
    df_market_scores
    .withColumn("question_lower", F.lower(F.col("question")))
    .withColumn(
        "topic_category",
        F.when(
            F.col("question_lower").rlike(
                "election|president|governor|senate|congress|prime minister|parliament|mayor|trump|biden|democrat|republican|labour|conservative|nominee|seats|assembly"
            ),
            "politics/elections"
        )
        .when(
            F.col("question_lower").rlike(
                "bitcoin|btc|ethereum|eth|crypto|usdt|usdc|solana|xrp|doge|memecoin|stablecoin|market cap"
            ),
            "crypto"
        )
        .when(
            F.col("question_lower").rlike(
                "fed|federal reserve|rate cut|interest rate|inflation|recession|gdp|cpi|unemployment|tariff|oil|gold|stock|s&p|nasdaq|bank of england|bps|trading day"
            ),
            "economy/finance"
        )
        .when(
            F.col("question_lower").rlike(
                "f1|formula 1|nba|nfl|mlb|nhl|champion|championship|world cup|premier league|ligue 1|bundesliga|la liga|serie a|tennis|ufc|boxing|olympics|racing|football|soccer|baseball|basketball|bears|orioles|red bull|drivers|constructors|assists|standings|top 4"
            ),
            "sports"
        )
        .when(
            F.col("question_lower").rlike(
                "openai|anthropic|google|meta|apple|microsoft|tesla|nvidia|ai model|chatgpt|gpt|claude|gemini|technology|spacex"
            ),
            "technology/ai"
        )
        .when(
            F.col("question_lower").rlike(
                "movie|album|song|grammy|oscar|emmy|netflix|disney|celebrity|actor|actress|music|box office|eurovision|billboard|artist|morgan wallen"
            ),
            "entertainment"
        )
        .when(
            F.col("question_lower").rlike(
                "war|ceasefire|iran|ukraine|russia|china|israel|gaza|nato|military|forces|peace|nobel|netanyahu|assange|guinea-bissau"
            ),
            "world events/geopolitics"
        )
        .when(
            F.col("question_lower").rlike(
                "covid|vaccine|disease|health|climate|earthquake|hurricane|temperature|science|space|nasa|arctic|sea ice"
            ),
            "science/health"
        )
        .otherwise("other")
    )
)

df_market_scores_categorized.groupBy("topic_category").agg(
    F.count("*").alias("market_count")
).orderBy(F.desc("market_count")).show(30, truncate=False)

## 13. Main Risk Driver

This section identifies the main reason each market was flagged.

In [ ]:
df_market_driver_counts = (
    df_scored
    .groupBy("market_id")
    .agg(
        F.sum("spread_spike_flag").alias("spread_spike_count"),
        F.sum("spread_instability_flag").alias("spread_instability_count"),
        F.sum("price_jump_flag").alias("price_jump_count"),
        F.sum("activity_spike_flag").alias("activity_spike_count"),
        F.sum("large_order_flag").alias("large_order_count"),
        F.sum("high_volume_change_flag").alias("high_volume_change_count"),
        F.sum("imbalance_flag").alias("imbalance_count"),
        F.sum("liquidity_stress_flag").alias("liquidity_stress_count")
    )
)

df_market_scores_explained = (
    df_market_scores_categorized
    .join(df_market_driver_counts, on="market_id", how="left")
    .withColumn(
        "main_risk_driver_value",
        F.greatest(
            "spread_spike_count",
            "spread_instability_count",
            "price_jump_count",
            "activity_spike_count",
            "large_order_count",
            "high_volume_change_count",
            "imbalance_count",
            "liquidity_stress_count"
        )
    )
    .withColumn(
        "main_risk_driver",
        F.when(F.col("main_risk_driver_value") == F.col("spread_spike_count"), "spread spike")
         .when(F.col("main_risk_driver_value") == F.col("spread_instability_count"), "spread instability")
         .when(F.col("main_risk_driver_value") == F.col("price_jump_count"), "price jump")
         .when(F.col("main_risk_driver_value") == F.col("activity_spike_count"), "activity spike")
         .when(F.col("main_risk_driver_value") == F.col("large_order_count"), "large order")
         .when(F.col("main_risk_driver_value") == F.col("high_volume_change_count"), "high volume change")
         .when(F.col("main_risk_driver_value") == F.col("imbalance_count"), "buy/sell imbalance")
         .when(F.col("main_risk_driver_value") == F.col("liquidity_stress_count"), "liquidity stress")
         .otherwise("none")
    )
)

## 14. Final Market Ranking and Category Summary

The final market ranking prioritizes sustained abnormal behavior using `high_risk_window_rate`, then average risk score.

In [ ]:
df_final_market_ranking = (
    df_market_scores_explained
    .select(
        "market_id",
        "question",
        "topic_category",
        "window_count",
        "avg_risk_score",
        "max_risk_score",
        "high_risk_windows",
        "very_high_risk_windows",
        "high_risk_window_rate",
        "overall_avg_spread",
        "max_price_range",
        "avg_imbalance_score",
        "total_anomaly_count",
        "main_risk_driver"
    )
    .orderBy(
        F.desc("high_risk_window_rate"),
        F.desc("avg_risk_score"),
        F.desc("max_risk_score"),
        F.desc("total_anomaly_count")
    )
)

df_final_market_ranking.show(30, truncate=False)

In [ ]:
df_final_category_summary = (
    df_market_scores_explained
    .groupBy("topic_category")
    .agg(
        F.count("*").alias("market_count"),
        F.avg("avg_risk_score").alias("avg_risk_score"),
        F.avg("max_risk_score").alias("avg_max_risk_score"),
        F.sum("high_risk_windows").alias("total_high_risk_windows"),
        F.sum("very_high_risk_windows").alias("total_very_high_risk_windows"),
        F.avg("high_risk_window_rate").alias("avg_high_risk_window_rate"),
        F.avg("overall_avg_spread").alias("avg_spread"),
        F.avg("avg_imbalance_score").alias("avg_imbalance_score")
    )
    .orderBy(F.desc("avg_risk_score"))
)

df_final_category_summary.show(30, truncate=False)

## 15. Visualizations

Convert only small final tables to Pandas for plotting.

In [ ]:
category_summary_pd = df_final_category_summary.toPandas()

top_markets_pd = (
    df_final_market_ranking
    .limit(20)
    .toPandas()
)

def shorten_text(text, max_len=70):
    if text is None:
        return ""
    return text if len(text) <= max_len else text[:max_len-3] + "..."

top_markets_pd["question_short"] = top_markets_pd["question"].apply(shorten_text)

In [ ]:
# Average risk score by category

plt.figure(figsize=(12, 6))

category_summary_pd_sorted = category_summary_pd.sort_values(
    "avg_risk_score",
    ascending=True
)

plt.barh(
    category_summary_pd_sorted["topic_category"],
    category_summary_pd_sorted["avg_risk_score"]
)

plt.xlabel("Average Risk Score")
plt.ylabel("Topic Category")
plt.title("Average Buyer Risk Score by Topic Category")
plt.tight_layout()
plt.show()

In [ ]:
# Total high-risk windows by category

plt.figure(figsize=(12, 6))

category_summary_pd_sorted = category_summary_pd.sort_values(
    "total_high_risk_windows",
    ascending=True
)

plt.barh(
    category_summary_pd_sorted["topic_category"],
    category_summary_pd_sorted["total_high_risk_windows"]
)

plt.xlabel("Total High-Risk Windows")
plt.ylabel("Topic Category")
plt.title("Total High-Risk Windows by Topic Category")
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 markets by sustained high-risk window rate

plt.figure(figsize=(12, 8))

top_markets_pd_plot = top_markets_pd.sort_values(
    "high_risk_window_rate",
    ascending=True
)

plt.barh(
    top_markets_pd_plot["question_short"],
    top_markets_pd_plot["high_risk_window_rate"]
)

plt.xlabel("High-Risk Window Rate")
plt.ylabel("Market Question")
plt.title("Top 20 Markets by Sustained High-Risk Window Rate")
plt.tight_layout()
plt.show()

## 16. Save Final Outputs

This section saves final outputs to a relative folder in the working directory.

In [ ]:
# Clean and create output folder

if os.path.exists(OUTPUT_PATH):
    shutil.rmtree(OUTPUT_PATH)

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save full Spark outputs as Parquet folders
df_final_market_ranking.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/final_market_ranking_parquet"
)

df_final_category_summary.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/final_category_summary_parquet"
)

# Save smaller CSV outputs for easy inspection
df_final_market_ranking.limit(1000).toPandas().to_csv(
    f"{OUTPUT_PATH}/final_market_ranking_sample.csv",
    index=False
)

df_final_category_summary.toPandas().to_csv(
    f"{OUTPUT_PATH}/final_category_summary.csv",
    index=False
)

print("Outputs saved to:", OUTPUT_PATH)

## 17. Interpretation and Limitations

This notebook built a Spark-based buyer-risk scoring pipeline for Polymarket orderbook data. The system aggregated tick-level orderbook records into 1-hour market windows and calculated abnormality signals based on bid-ask spread behavior, price movement, orderbook activity, large order changes, buy/sell imbalance, and liquidity stress.

The final output ranks markets using a Buyer Risk / Manipulation-Likelihood Score. The score is not a fraud label and does not prove manipulation. Instead, it identifies markets with unusual orderbook behavior that may require further review.

The analysis has several limitations:

1. Only markets that successfully joined with the labels table were included.
2. The MVP uses one orderbook file, so results reflect one sample period only.
3. Topic categories were created using keyword rules because the original category field was mostly blank.
4. The score is rule-based and should be treated as a screening tool, not a definitive manipulation-detection system.

Future improvements may include processing multiple orderbook days, adding unsupervised machine learning such as clustering or Isolation Forest, improving topic classification using NLP, and validating flagged markets against external news or known market events.